# StarCoder: Fine-tuning Llama 2 for Go and Rust Generation

This example demonstrates how to fine-tune Meta's Llama-2-7B model to
generate high-quality Go and Rust code. While Llama 2 is a powerful
general-purpose language model, it can struggle with generating
syntactically correct and idiomatic code in specific programming
languages. This training setup aims to enhance its coding capabilities
specifically for Go and Rust.

## Overview

The training pipeline consists of three main stages:

1. Dataset ingestion from the StarCoder dataset
2. Multi-node distributed training using PyTorch FSDP
3. Model evaluation on coding tasks

Unlike the other framework examples, `torchrun` and `hf_accelerate`
aren't tied to a specific training framework — they just spawn a
user-provided Python script under `torchrun` / `accelerate launch`
across the Modal cluster. We demo both launchers below with the same
training script so you can compare.

## Prerequisites

- Modal account with access to H100 GPUs
- Hugging Face account with access to Llama 2 model family
- Weights & Biases account (optional, for experiment tracking)

In [ ]:
! pip install -q git+https://github.com/modal-projects/training-gym.git@joy/initial-setup

In [ ]:
import modal

from modal_training_gym.common.dataset import DatasetConfig
from modal_training_gym.common.models import BaseModelType, Model
from modal_training_gym.common.wandb import WandbConfig
from modal_training_gym.frameworks.hf_accelerate import (
    AccelerateConfig,
    AccelerateModalConfig,
    build_accelerate_app,
)
from modal_training_gym.frameworks.torchrun import (
    DATASET_MOUNT_PATH,
    TorchrunConfig,
    TorchrunModalConfig,
    build_torchrun_app,
)

## 1. Dataset Preparation

Download and prepare the StarCoder dataset (Go + Rust arrow files).
The `prepare()` method on `StarcoderGoRustDataset` below does what the
reference `download_dataset.py::ingest_dataset` does: list files from
`bigcode/starcoderdata`, filter out incompatible shards, and
`snapshot_download` them into the data volume.

- Downloads code samples from the StarCoder dataset
- Processes and validates the data
- Stores it in a Modal volume for training

In [ ]:
class StarcoderGoRustDataset(DatasetConfig):
    """StarCoder: Go + Rust shards from `bigcode/starcoderdata`."""

    def __init__(self, data_path):
        self._data_path = str(data_path)
        # The train script globs `go/**/*.arrow` + `rust/**/*.arrow`
        # under `--data_dir`, so we point prompt_data at the root.
        self.prompt_data = self._data_path

    def prepare(self):
        import os
        from datasets import load_dataset
        from huggingface_hub import HfApi

        api = HfApi()
        all_paths = api.list_repo_files(
            repo_id="bigcode/starcoderdata", repo_type="dataset"
        )
        # Only Go and Rust parquet shards.
        excluded = (
            "jupyter-scripts-dedup-filtered",
            "jupyter-structured-clean-dedup",
            "github-issues-filtered-structured",
            "git-commits-cleaned",
        )
        file_paths = [
            p
            for p in all_paths
            if ".parquet" in p
            and any(p.startswith(f"{lang}/") for lang in ("go", "rust"))
            and not any(x in p for x in excluded)
        ]
        print(f"Downloading {len(file_paths)} shards…")
        for fp in file_paths:
            save_path = f"{self._data_path}/{fp}"
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            ds = load_dataset("bigcode/starcoderdata", data_files=fp)
            ds.save_to_disk(save_path)
        print(f"Saved {len(file_paths)} shards to {self._data_path}")

## 2. Training Script

The training script is the same for both launchers — supervised
fine-tuning of Llama-2-7B on packed Go + Rust sequences via TRL's
`SFTTrainer` and PyTorch FSDP (`full_shard auto_wrap`, activation
checkpointing on). We define it as a single Python string; the
framework's `upload_script` writes it to a scripts volume that the
train function then mounts.

In [ ]:
import textwrap

# Indented to match the function body so `textwrap.dedent` can strip
# the common leading whitespace uniformly.
TRAIN_SCRIPT = textwrap.dedent(r'''
    import argparse
    import os
    from pathlib import Path

    import datasets
    import torch
    import torch.distributed as dist
    from datasets import load_dataset
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        DataCollatorForLanguageModeling,
    )
    from trl import SFTConfig, SFTTrainer

    MODEL_NAME = "meta-llama/Llama-2-7b-hf"


    def download_llama2(cache_dir):
        token = os.environ["HF_TOKEN"]
        tok = AutoTokenizer.from_pretrained(MODEL_NAME, token=token, cache_dir=cache_dir)
        tok.pad_token = tok.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, torch_dtype="auto", token=token, cache_dir=cache_dir,
        )
        model.config.use_cache = False  # KV cache breaks activation checkpointing
        return model, tok


    def build_packed_ds(data_dir, tokenizer, buffer_size, block):
        eos_id = tokenizer.eos_token_id

        def gen():
            files = []
            for lang_dir in ["go", "rust"]:
                files.extend(str(f) for f in (data_dir / lang_dir).glob("**/*.arrow"))
            print(f"Found {len(files)} arrow files")
            ds = load_dataset("arrow", data_files=files, split="train", streaming=True)
            ds = ds.shuffle(buffer_size=buffer_size, seed=44)

            buf = []
            for rec in ds:
                buf.extend(
                    tokenizer(rec["content"], add_special_tokens=False)["input_ids"]
                    + [eos_id]
                )
                while len(buf) >= block:
                    yield {"input_ids": buf[:block], "attention_mask": [1] * block}
                    del buf[:block]

        return datasets.IterableDataset.from_generator(gen)


    def parse_args():
        p = argparse.ArgumentParser()
        p.add_argument("--data_dir", required=True)
        p.add_argument("--output_dir", required=True)
        p.add_argument("--model_cache_dir", default=None)
        p.add_argument("--epochs", type=int, default=2)
        p.add_argument("--batch_per_device", type=int, default=16)
        p.add_argument("--grad_accum", type=int, default=2)
        p.add_argument("--block_size", type=int, default=4096)
        return p.parse_args()


    def main():
        args = parse_args()

        if not dist.is_initialized():
            dist.init_process_group(
                backend="nccl",
                device_id=torch.device(f"cuda:{os.environ['LOCAL_RANK']}"),
            )

        model, tokenizer = download_llama2(args.model_cache_dir)
        train_ds = build_packed_ds(
            Path(args.data_dir), tokenizer, buffer_size=20_000, block=args.block_size,
        )
        collator = DataCollatorForLanguageModeling(tokenizer, mlm=False, pad_to_multiple_of=8)

        cfg = SFTConfig(
            output_dir=args.output_dir,
            seed=1234,
            per_device_train_batch_size=args.batch_per_device,
            gradient_accumulation_steps=args.grad_accum,
            learning_rate=8e-5,
            lr_scheduler_type="cosine",
            warmup_ratio=0.03,
            bf16=True,
            max_seq_length=args.block_size,
            save_steps=125,
            logging_steps=1,
            report_to="wandb" if os.environ.get("WANDB_PROJECT") else "none",
            run_name=os.environ.get("WANDB_RUN_NAME"),
            fsdp="full_shard auto_wrap",
            fsdp_config={
                "activation_checkpointing": True,
                "forward_prefetch": False,
                "fsdp_transformer_layer_cls_to_wrap": ["LlamaDecoderLayer"],
            },
            num_train_epochs=args.epochs,
            max_steps=10000,
        )

        SFTTrainer(
            model=model,
            train_dataset=train_ds,
            args=cfg,
            data_collator=collator,
        ).train()

        if dist.is_initialized():
            dist.destroy_process_group()


    if __name__ == "__main__":
        main()
''').lstrip("\n")

## 3. Training Config

Shared settings for both launchers — matches the reference:

| Parameter                | Value                   |
|--------------------------|-------------------------|
| Global batch size        | 2048                    |
| Per-device batch size    | 16                      |
| Gradient accumulation    | 2                       |
| Learning rate            | 8e-5 (cosine decay)     |
| Context length           | 4096 tokens             |
| Epochs                   | 2                       |
| Nodes × GPUs             | 2 × 8 H100              |

This command:

- Launches a cluster of 2 nodes with 8 H100 GPUs each
- Uses PyTorch FSDP (Fully Sharded Data Parallel) for efficient
  distributed training
- Automatically configures RDMA for high-speed inter-node communication
- Saves checkpoints periodically to a Modal volume

In [ ]:
_MODEL = Model(BaseModelType.Llama2_7B)
_DATASET = StarcoderGoRustDataset(DATASET_MOUNT_PATH)
_WANDB = WandbConfig(project="bigcode-starcoderdata-training")

_SHARED_SCRIPT_ARGS = [
    "--epochs", "2",
    "--batch_per_device", "16",
    "--grad_accum", "2",
    "--block_size", "4096",
    "--model_cache_dir", "/model/model_cache",
]

## 4a. torchrun launcher

Launches the training script via `torchrun` — the stock PyTorch
distributed launcher. Directly invokes
`torch.distributed.run.run(...)` on every node.

In [ ]:
class _Torchrun(TorchrunConfig):
    model = _MODEL
    dataset = _DATASET
    wandb = _WANDB

    n_nodes = 2
    gpus_per_node = 8

    train_script_source = TRAIN_SCRIPT
    script_args = _SHARED_SCRIPT_ARGS

torchrun_app = build_torchrun_app(
    modal=TorchrunModalConfig(gpu="H100"),
    config=_Torchrun(),
)

## 4b. HuggingFace Accelerate launcher

Same script, launched via `accelerate launch` — HF Accelerate adds
mixed-precision setup (`--mixed_precision bf16`) and some other
niceties on top of torchrun's spawn semantics.

In [ ]:
class _Accelerate(AccelerateConfig):
    model = _MODEL
    dataset = _DATASET
    wandb = _WANDB

    n_nodes = 2
    gpus_per_node = 8

    train_script_source = TRAIN_SCRIPT
    script_args = _SHARED_SCRIPT_ARGS
    mixed_precision = "bf16"

accelerate_app = build_accelerate_app(
    modal=AccelerateModalConfig(gpu="H100"),
    config=_Accelerate(),
)

## 5. Run it

Each app has the same three functions:

- `download_dataset` — runs `StarcoderGoRustDataset.prepare()`.
- `upload_script`    — writes `TRAIN_SCRIPT` into the scripts volume.
- `train`            — clustered multi-node, launches the uploaded script.

Both apps use their own volumes (`<app_name>-data`, `-model`,
`-scripts`), so running one doesn't affect the other.

Key training parameters (configurable on the `_Torchrun` /
`_Accelerate` classes above):

- Global batch size: 2048
- Per-device batch size: 16
- Gradient accumulation steps: 2
- Learning rate: 8e-5 with cosine decay
- Context length: 4096 tokens

### Interactive — torchrun

In [ ]:
with torchrun_app.run():
    torchrun_app.download_dataset.remote()
    torchrun_app.upload_script.remote()
    torchrun_app.train.remote()

### Interactive — accelerate

In [ ]:
with accelerate_app.run():
    accelerate_app.download_dataset.remote()
    accelerate_app.upload_script.remote()
    accelerate_app.train.remote()

## Performance Monitoring

If you've configured Weights & Biases:

- Training metrics are logged in real-time
- You can monitor loss, learning rate, and GPU utilization
- Compare different training runs and hyperparameters

## Scaling

Sample consumption scales with the number of nodes and GPUs. Scaling
is sublinear but can be improved by increasing the global batch size
and tuning FSDP configurations.

| Nodes | GPUs | Samples per minute | Tokens per minute |
|-------|------|--------------------|-------------------|
| 2     | 8    | 2,898              | 6,151,645         |
| 4     | 8    | 4,981              | 10,625,570        |
| 8     | 8    | 7,675              | 16,357,785        |

## Customization

- Adjust the number of nodes and GPUs on the `_Torchrun` / `_Accelerate`
  subclasses (`n_nodes`, `gpus_per_node`).
- Change training hyperparameters inside `TRAIN_SCRIPT`.
- Add new evaluation prompts by writing a new framework function.
- Configure data preprocessing in `StarcoderGoRustDataset.prepare()`.